# Suite CON — Predicates

A predicate is a rule over symbols, each bound to an object of a schema, and parameters, each a value; it applies to a
match, which binds every symbol. Predicates are terms of the predicate algebra, built fluently as schemas are, and are
data: applied by reference, written with their symbols' schema names, and read back through a store that resolves them.

In [ ]:
from mbse.Expressions import Expressions as E
from mbse.Expressions.Dialects.Python import Text
from mbse.Expressions.Framework import Terms
from mbse.Patterns import Constraints as C, Predicates as P, register
from mbse.Schemas.Framework import JSON, Proxies, Schemas as S, Validators as SV
from support import Contact, Phone, book, raises

the = E.variable("the")
IsAnAdult = (
    P.OfPredicate.Builder()
    .name("IsAnAdult")
    .description("18 or older")
    .symbols({"the": Contact})
    .requires(Text.FromFunction(lambda the: the.age >= 18))
    .create()
)
Numbered = P.OfPredicate.Builder().name("Numbered").symbols({"the": Phone}).requires(the.has("number")).create()

## CON-01 · A predicate is built fluently: a name, a description, symbols, parameters and conditions

In [ ]:
assert (IsAnAdult.name, IsAnAdult.description, list(IsAnAdult.symbols), IsAnAdult.parameters) == (
    "IsAnAdult", "18 or older", ["the"], {})
assert IsAnAdult.symbols["the"] is Contact and isinstance(IsAnAdult.rule, E.OfOperation.Data)  # read from the lambda
assert Terms.same(IsAnAdult.rule, the.age.ge(18).data)  # the same rule as the writer builds
assert Numbered.description is None and IsAnAdult.validate(core=True) == [] and IsAnAdult.binds() == ("the",)
two = P.OfPredicate.Builder().symbols({"a": Contact}).symbols({"b": Phone}).parameters(
    lambda p: p.name("n").of(lambda t: t.as_native(int)), lambda p: p.name("any")).create()
assert list(two.symbols) == ["a", "b"] and two.rule is None  # symbols add up, in order; nothing validates yet
assert list(two.parameters) == ["n", "any"] and two.parameters["any"] is None and two.binds() == ("a", "b", "n", "any")
renamed = P.OfPredicate.Builder(IsAnAdult).name("IsGrown").clone()
assert renamed.name == "IsGrown" and IsAnAdult.name == "IsAnAdult" and renamed.rule is IsAnAdult.rule
assert renamed.symbols == IsAnAdult.symbols and renamed.symbols is not IsAnAdult.symbols
assert P.OfPredicate.Builder(renamed).description("grown").update() is renamed and renamed.description == "grown"
both = P.OfPredicate.Builder(IsAnAdult).requires(the.has("name")).forbids(the.has("pager")).clone()  # conditions add up
assert Terms.same(both.rule, the.age.ge(18).and_(the.has("name")).and_(the.has("pager").not_()).data)
assert IsAnAdult.rule is not both.rule
assert Terms.same(IsAnAdult, P.OfPredicate.Builder(IsAnAdult).clone())  # equal symbols compare equal
assert not Terms.same(IsAnAdult, P.OfPredicate(IsAnAdult.name, IsAnAdult.description, IsAnAdult.rule, {"the": Phone}))
assert not Terms.same(IsAnAdult, P.OfPredicate(IsAnAdult.name, IsAnAdult.description, IsAnAdult.rule, {"x": Contact}))
assert not Terms.same(IsAnAdult, P.OfPredicate.Builder(IsAnAdult).symbols({"x": Phone}).clone())  # one more
builder = P.OfPredicate.Builder().symbols({"c": Contact}).parameters(lambda p: p.name("least"))
assert Terms.same(builder.c.data, E.variable("c").data) and Terms.same(builder.least.data, E.variable("least").data)  # declared
with raises(AttributeError, match="nope"):
    builder.nope
assert P.OfPredicate.resolve(IsAnAdult) is IsAnAdult
assert P.OfPredicate.resolve(lambda b: b.name("X").symbols({"x": Contact}).requires(True)).name == "X"
with raises(TypeError, match="expected a predicate or a callable taking its builder, got 'X'"):
    P.OfPredicate.resolve("X")
with raises(ValueError, match="create() is only valid without a source instance; use clone() or update()"):
    P.OfPredicate.Builder(IsAnAdult).create()
with raises(ValueError, match="clone() is only valid with a source instance"):
    P.OfPredicate.Builder().clone()
with raises(ValueError, match="update() is only valid with a source instance"):
    P.OfPredicate.Builder().update()
assert (IsAnAdult.schema_name(), IsAnAdult.owner(), IsAnAdult.identity() == IsAnAdult.identity()) == (
    "Patterns.Predicate", None, True)

## CON-02 · Predicates are checked when asked

In [ ]:
Value = S.OfObject.Builder().name("Value").create()
bad = P.OfPredicate.Builder().name("Bad").symbols({"a": Value, "b": S.OfObject.Builder().ref().create()}).requires(
    E.operation("pow", E.variable("a"), E.variable("n"))).create()
assert bad.validate(core=True) == ["symbol 'a' needs a named reference object schema",
                                   "symbol 'b' needs a named reference object schema",
                                   "rule: 'pow' is not a core operation",
                                   "rule: argument 0: variable 'a' is not bound",  # a symbol with a problem binds nothing
                                   "rule: argument 1: variable 'n' is not bound"]
assert P.OfPredicate.Builder().create().validate() == ["a predicate needs a rule"]  # a name and symbols are optional
with raises(ValueError, match="predicate 'Bad': symbol 'a' needs a named reference object schema; predicate 'Bad'"):
    C.check([bad])
with raises(ValueError, match="predicate 'IsAnAdult': defined twice; predicate None: a predicate of a set needs a name"):
    C.check([IsAnAdult, IsAnAdult, lambda b: b.symbols({"x": Contact}).requires(True)])
with raises(ValueError, match="a set's arguments are predicates"):
    C.check(C.OfSet((the.data,)))
predicates = C.check([IsAnAdult, lambda b: b.name("Named").symbols({"the": Contact}).requires(the.has("name"))])
assert isinstance(predicates, C.OfSet) and C.check(predicates) is predicates

## CON-03 · A set orders predicates

In [ ]:
predicates = C.OfSet.Builder().predicates(IsAnAdult).predicates(Numbered).create()
assert predicates.predicates == (IsAnAdult, Numbered)  # a tuple, in order
assert C.OfSet.Builder().create().predicates == () and predicates.schema_name() == "Patterns.Set"
assert predicates.owner() is None and predicates.identity() == predicates.identity() != C.OfSet.Builder().create().identity()
grown = C.OfSet.Builder(predicates).predicates(lambda b: b.name("Grown").symbols({"the": Contact}).requires(True)).clone()
assert [p.name for p in grown.predicates] == ["IsAnAdult", "Numbered", "Grown"] and len(predicates.predicates) == 2
assert C.OfSet.Builder(grown).predicates(IsAnAdult).update() is grown and len(grown.predicates) == 4
with raises(ValueError, match="create() is only valid without a source instance; use clone() or update()"):
    C.OfSet.Builder(predicates).create()

## CON-04 · Predicates are data: written by their symbols' schema names, read back through a store

In [ ]:
shared = the.age.ge(18).data
predicates = C.OfSet.Builder().predicates(
    P.OfPredicate.Builder().name("IsAnAdult").description("18 or older").symbols({"the": Contact}).requires(shared).create(),
    lambda b: b.name("Owns").symbols({"c": Contact, "p": Phone}).requires(E.operation("implies", shared, E.variable("p").has("number"))),
).create()
store = C.OfStore(book())  # resolves the symbols' schemas in the address book
text = JSON.ToJSON(store).Reachable(C.OfSet.Schema, predicates)
assert '"symbols": [{"name": "c", "type": {"named": {"name": "Contact"}}}' in text  # by name: no store needed to write
assert text.count('"kind": "operation"') == 4  # get, ge, implies and has: the shared rule's two are written once
assert JSON.ToJSON(C.Builders).Reachable(C.OfSet.Schema, predicates) == text  # any store writes them alike
copy = JSON.FromJSON(store).Reachable(C.OfSet.Schema, text)
assert [(p.name, p.description) for p in copy.predicates] == [("IsAnAdult", "18 or older"), ("Owns", None)]
assert copy.predicates[1].symbols["c"] is Contact and copy.predicates[1].symbols["p"] is Phone  # the store's own
assert copy.predicates[1].rule.arguments[0] is copy.predicates[0].rule  # still shared
assert Terms.same(copy.predicates[1].rule, predicates.predicates[1].rule)
assert SV.Validate(store).Reachable(C.OfSet.Schema, predicates) == []  # valid data, by mbse-schemas' rules
with raises(LookupError, match="no schema named 'Contact' in the module or the store"):
    JSON.FromJSON(C.Builders).Reachable(C.OfSet.Schema, text)  # reading resolves names, so it needs the store
Inline = S.OfObject.Builder().ref().properties(lambda p: p.name("n").of(lambda t: t.as_native(int))).create()
loose = P.OfPredicate.Builder().name("Loose").symbols({"x": Inline}).parameters(lambda p: p.name("k")).requires(True).create()
back = JSON.FromJSON(C.Builders).Reachable(P.OfPredicate.Schema, JSON.ToJSON(C.Builders).Reachable(P.OfPredicate.Schema, loose))
assert back.symbols["x"] is not Inline and list(back.symbols["x"].properties) == ["n"]  # an unnamed schema, inline
assert back.parameters == {"k": None}  # a parameter of any type
built = store.builder("Patterns.Predicate", IsAnAdult)
built.property("name", lambda p: p.value(lambda a: a.as_native(lambda n: n.set("Grown"))))
assert built.update() is IsAnAdult and IsAnAdult.name == "Grown" and IsAnAdult.symbols["the"] is Contact
IsAnAdult.name = "IsAnAdult"

## CON-05 · Decoding refuses what a predicate or a set cannot hold

In [ ]:
store = C.OfStore(book())


def entry(target, i):
    def fill(e):
        if target is not None:
            e.link("argument", lambda l: l.set(target))
        e.property("index", lambda p: p.value(lambda v: v.as_native(lambda n: n.set(i))))
    return lambda a: a.add(fill)


with raises(ValueError, match="a predicate's rule is argument 0, got index 1"):
    store.builder("Patterns.Predicate").adjacency("arguments", entry(the.data, 1)).create()
with raises(ValueError, match="link 'argument' is not set"):
    store.builder("Patterns.Predicate").adjacency("arguments", entry(None, 0)).create()
odd = store.builder("Patterns.Set").adjacency("arguments", entry(the.data, 0)).create()
assert odd.predicates == (the.data,) and odd.check() == ["a set's arguments are predicates"]  # read, then refused
blank = store.builder("Patterns.Predicate").create()  # what a snapshot leaves out is absent, and validate() says so
assert (blank.name, blank.description, blank.symbols, blank.parameters, blank.rule) == (None, None, {}, {}, None)
assert JSON.ToJSON(C.Builders)(P.OfPredicate.Schema, blank) == '{"root": "s0", "objects": {"s0": {"kind": "predicate"}}}'

## CON-06 · Another store holds predicates as proxies

In [ ]:
store = register(Proxies.OfStore())
assert {"Patterns.Predicate", "Patterns.Set", "Patterns.OfApply", "Patterns.OfChoices", "Expressions.OfOperation"} <= set(store.names())
assert register(store) is store  # what it holds already is skipped
loose = P.OfPredicate.Builder().name("Any").symbols({"x": S.OfObject.Builder().ref().create()}).requires(True).create()
proxy = JSON.FromJSON(store).Reachable(P.OfPredicate.Schema, JSON.ToJSON(C.Builders).Reachable(P.OfPredicate.Schema, loose))
assert proxy.schema_name() == "Patterns.Predicate" and Proxies.store_of(proxy) is store and proxy.name == "Any"

## CON-07 · A predicate is applied by reference, its arguments bound to its symbols and parameters

In [ ]:
HasName = (P.OfPredicate.Builder().name("HasName").symbols({"person": Contact}).parameters(lambda p: p.name("name"))
           .requires(Text.FromFunction(lambda person, name: person.name == name)).create())
Named = P.OfPredicate.Builder().name("Named").symbols({"c": Contact})
Ann = Named.requires(HasName(Named.c, "ann")).create()  # the builder gives its symbols as variables
applied = Ann.rule
assert isinstance(applied, P.OfApply) and applied.predicate is HasName  # the predicate itself
assert Terms.same(applied.arguments[0], E.variable("c").data) and applied.arguments[1].value == "ann"
assert Terms.same(HasName.call(E.variable("c"), "ann").arguments[1], applied.arguments[1])  # call is the same
assert Ann.validate(core=True) == [] and HasName.validate(core=True) == []
assert P.DIALECT.validate(HasName(E.variable("c")), bound={"c"}) == ["'HasName' takes 2 arguments, got 1"]
assert P.DIALECT.validate(P.OfApply(the.data, ()), bound={"the"}) == ["an application's predicate must be a predicate"]
assert P.DIALECT.validate(P.OfApply(None, ())) == ["an apply needs a predicate"]
store = book()
ann, bob = store.Contact().name("ann").create(), store.Contact().name("bob").create()
evaluate = P.Evaluator(store)
assert P.holds(evaluate, Ann.rule, {"c": ann}) is True and P.holds(evaluate, Ann.rule, {"c": bob}) is False
assert evaluate(HasName(E.variable("c"), E.variable("n")), {"c": bob, "n": "bob"}) is True
shared = C.OfSet.Builder().predicates(Ann, lambda b: b.name("Bob").symbols({"c": Contact}).requires(HasName(E.variable("c"), "bob"))).create()
schemas = C.OfStore(book())
text = JSON.ToJSON(schemas).Reachable(C.OfSet.Schema, shared)
assert text.count('"name": "HasName"') == 1  # applied twice, written once
copy = JSON.FromJSON(schemas).Reachable(C.OfSet.Schema, text)
assert copy.predicates[0].rule.predicate is copy.predicates[1].rule.predicate and copy.predicates[0].rule.predicate.parameters == {"name": None}
cyclic = P.OfPredicate.Builder().name("Loop").symbols({"c": Contact}).create()
P.OfPredicate.Builder(cyclic).requires(cyclic(E.variable("c"))).update()
assert P.DIALECT.validate(cyclic) == ["rule: predicate: the expression contains a cycle"]